<a href="https://colab.research.google.com/github/bsalami-092/Data_Science_Journey_Documentation/blob/main/Health_Facilities_Service_Area_Determination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To determine the Service Area of Health Facilities in one LGA in Sokoto State using Nominatim for geocoding and OpenRouteService(ORS) API for service area generation in Python. Follow this comprehensive approach:

This giude will walk you through how to determine the 2km and 5km service areas for health facilities in one LGA  of Sokoto State using Nominatim for geocoding and OpenRouteService(ORS) for service area generation in Python.

In [ ]:
!pip install openrouteservice

In [ ]:
# Step 1: Import the required libraries
import geopandas as gpd
from shapely.geometry import Point, shape
import pandas as pd
import folium
import time
from  google.colab  import files, output
import openrouteservice as ors

# Step 2: Load Free Health Facility Data

In [ ]:
# Upload the shapefile components
uploaded = files.upload()



In [ ]:
# Load health facilities data
df = pd.read_csv('GRID3_NGA_health_facilities_v2_0_5806009649412052847.csv')

df.head()

In [ ]:
df.info()

Convert Latitude and Longitude to Numeric

If your Latitude and Longitude columns contain string values, convert them to numbers.

# Visualize and Map the data

* Convert  Data to GeoDataFrame (for spatial processing)

In [ ]:
# Create geometry column
geometry = [Point(xy) for xy in zip(df.longitude, df.latitude)]

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)


gdf.head()

In [ ]:
gdf_sokoto = gdf[gdf['state'] == 'Sokoto']

gdf_sokoto.head()

In [ ]:
gdf_sokoto.info()

Step 5: Generate Service Area Using OpenRouteService(ORS)

* ORS allows us to generate service areas(isochrones) around health facilities. We will use  2km and 5km travel distances

In [ ]:
# Setup ORS Client
from google.colab import userdata

API_KEY = userdata.get('ORS_API_KEY')
client = ors.Client(key=API_KEY)

# Generate Service Area

In [ ]:
# Test service area for one facility
coordinates = [[5.25, 13.05]] # Sokoto Example [Longitude, Latitude]
isochrones = client.isochrones(
    locations=coordinates,
    profile='driving-car',
    range=[120, 300] # 2km(~120 secs) and 5km(~300 secs)

)


print(isochrones)

In [ ]:
# Function to generate Service Area
def get_service_area(lon, lat):
  if not (-180 <= lon <= 180 and -90 <= lat <= 90):
    print(f'Invalid coordinates: {lon}, {lat}')
    return None, None
  try:
    isochrones = client.isochrones(
        locations=[[lon, lat]],
        profile='driving-car',
        range=[120, 300]
    )
    return shape(isochrones['features'][0]['geometry']), shape(isochrones['features'][1]['geometry'])
  except Exception as e:
    print(f'Error for {lon}, {lat}: {e}')
    return None, None

Apply the function to generate service area

In [ ]:
# Work with a smaller subset (first 20 points)
subset = gdf_sokoto.iloc[:20].copy()

service_areas_2km = []
service_areas_5km = []

# Loop through subset
for _, row in subset.iterrows():
    polygon_2km, polygon_5km = get_service_area(row['longitude'], row['latitude'])
    service_areas_2km.append(polygon_2km)
    service_areas_5km.append(polygon_5km)
    time.sleep(1)  # prevent rate limit

# ✅ Add results to subset (not full gdf_sokoto)
subset['service_area_2km'] = service_areas_2km
subset['service_area_5km'] = service_areas_5km

# ✅ Convert to GeoDataFrames
service_areas_2km_gdf = gpd.GeoDataFrame(subset, geometry='service_area_2km', crs=gdf_sokoto.crs)
service_areas_5km_gdf = gpd.GeoDataFrame(subset, geometry='service_area_5km', crs=gdf_sokoto.crs)

# View sample
service_areas_2km_gdf.head()


Visualize Service Areas on an Interactive Map

In [ ]:
# Create Map Centered on Sokoto
m = folium.Map(location=[13.0, 5.2], zoom_start=10)

# Add health facilities
for _, row in gdf.iterrows():
  folium.Marker(
      location=[row['latitude'], row['longitude']],
      popup=row['facility_level_option'],
      icon=folium.Icon(color ='green', icon='hospital')
      ).add_to(m)

  # Add 2km service area (Blue)
for _,row in service_areas_2km_gdf.iterrows(): # Corrected variable name and method
  if row.geometry:
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {'color': 'blue', 'fillOpacity': 0.3}
        ).add_to(m)

  # Add 5km service area (Red)
for _,row in service_areas_5km_gdf.iterrows(): # Corrected variable name and method
  if row.geometry:
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {'color': 'red', 'fillOpacity': 0.2}
        ).add_to(m)

 # Save and display the map
m.save("service_area_map_html")

output.enable_custom_widget_manager()
m


Save 2km and 5km seperately

In [ ]:
# Save 2km service area
service_areas_2km_gdf.drop(columns=['geometry', 'service_area_5km']).to_file('service_areas_2km.geojson', driver='GeoJSON')

# Save 5km service area
service_areas_5km_gdf.drop(columns=['geometry', 'service_area_2km']).to_file('service_areas_5km.geojson', driver='GeoJSON')